# MyAI K4 — SFT on 100M base (GPU, ~1h)
Base weights + 32k tokenizer come from K3 output (attached input, discovered via find). Dolly-15k + OASST pairs, masked SFT x500 steps (~2 epochs), val + frozen-set eval. v2 fixes: fully-masked blocks dropped (val NaN root cause).

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!git clone https://github.com/aloksingh9667/llm.git myai 2>&1 | tail -1
%cd /kaggle/working/myai
!pip install -q -r requirements/train.txt 2>&1 | tail -1
!find /kaggle/input -name 'final.pt' 2>/dev/null
!find /kaggle/input -name 'tokenizer-32k.json' 2>/dev/null

In [ ]:
import shutil, subprocess
base = subprocess.run(['find', '/kaggle/input', '-name', 'final.pt'], capture_output=True, text=True).stdout.strip().split('\n')[0]
tok = subprocess.run(['find', '/kaggle/input', '-name', 'tokenizer-32k.json'], capture_output=True, text=True).stdout.strip().split('\n')[0]
assert base.endswith('final.pt') and tok.endswith('.json'), 'K3 outputs missing'
shutil.copy(base, '/kaggle/working/base-100m.pt')
shutil.copy(tok, 'data/tokenizer_corpus/tokenizer-32k.json')
print('base + tokenizer staged')

In [ ]:
!python scripts/fetch_sft.py --source dolly --max-pairs 15000
!python scripts/fetch_sft.py --source oasst1 --max-pairs 10000
!python scripts/fetch_sft.py --source seed
!cat data/sft/dolly.jsonl data/sft/oasst1.jsonl data/sft/seed.jsonl > data/sft/combined.jsonl && wc -l data/sft/combined.jsonl

In [ ]:
!python scripts/pack_sft.py --pairs data/sft/combined.jsonl --tokenizer data/tokenizer_corpus/tokenizer-32k.json --seq-len 512 --out-dir data/processed/sft-32k

In [ ]:
!python scripts/train.py --model-config configs/myai-100m.yaml --data-dir data/processed/sft-32k --sft --lr 2e-5 --max-steps 500 --batch-size 4 --accum 8 --amp --eval-every 250 --ckpt-every 500 --init-from /kaggle/working/base-100m.pt --out-dir /kaggle/working/ckpts-100m-sft --report /kaggle/working/train-sft.json

In [ ]:
!python scripts/eval_model.py --checkpoint /kaggle/working/ckpts-100m-sft/final.pt --data-dir data/processed/sft-32k --report /kaggle/working/eval-sft.json
!python scripts/eval_sft.py --checkpoint /kaggle/working/ckpts-100m-sft/final.pt --tokenizer data/tokenizer_corpus/tokenizer-32k.json --report /kaggle/working/sft-frozen.json
!ls -la /kaggle/working/ckpts-100m-sft/ && cat /kaggle/working/train-sft.json